In [ ]:
# Fix protobuf version mismatch for kaggle_benchmarks
!pip install protobuf==5.29.6 --quiet


# Trinity Executive Function Battery — Working Memory Span

**Task 19 of 25** | Track 4: Executive | Brain Zone: DLPFC

This notebook measures working memory capacity.

## Trinity Neuroanatomical Foundation

DLPFC maintains φ-scaling capacity (3, 5, 8, 13, 21).

### Ternary Scoring {-1, 0, +1}

- **+1**: Correct
- **0**: Partial
- **-1**: Wrong

### φ-Scaling

Item counts: 3, 5, 8, 13, 21

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd

df = pd.read_csv('../../data/tefb_executive.csv')
memory_df = df[df['task'] == 'Working Memory Span'].copy()

In [ ]:
@dataclass
class MemoryResponse:
    answer: str
    confidence: float
    items_retained: int

    def ternary_score(self, expected: str) -> int:
        if self.answer == expected:
            return 1
        elif 0.5 < self.confidence:
            return 0
        else:
            return -1

In [ ]:
@kbench.task(name="trinity_dlpfc_working_memory")
def working_memory(llm, data, expected, item_count) -> float:
    prompt = f""Data: {data}\nOperations: Remember first, count vowels in second, check third...\nExpected answer: {expected}\nHow many items did you retain?\nConfidence (0-1)"
    response = llm.prompt(prompt, schema=MemoryResponse)
    ternary = response.ternary_score(expected)
    accuracy = 1.0 if response.answer == expected else -1.0
    retention_score = min(1.0, response.items_retained / item_count)
    final_score = 0.6 * accuracy + 0.2 * retention_score
    return max(-1.0, min(1.0, final_score))

print("Task registered")

In [ ]:
results = working_memory.evaluate(llm=[kbench.llm], evaluation_data=memory_df.head(10))
print(results.head())

In [ ]:
# Full eval
# results = working_memory.evaluate(llm=[kbench.llm], evaluation_data=memory_df)
# kbench.submit(task=working_memory, results=results, message="Trinity DLPFC Working Memory v1.0")
# print("Submitted!")